# 02. Treinar Splink (link_only Censo × CPF)

Profile, blocking pré-treino, treino do modelo, predict e clustering.

Usa `link_type='link_only'`: só gera pares **entre** Censo e CPF (não
deduplica dentro de cada base). A coorte não entra aqui — o modelo é treinado
sem ver os rótulos, e a avaliação roda em
[`03_validar_coorte.ipynb`](03_validar_coorte.ipynb) sobre o modelo salvo.

**EDA descritiva:** [`01_analise_descritiva.ipynb`](01_analise_descritiva.ipynb).


In [1]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    OUTPUT_DIR,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    SPLINK_PREDICTIONS,
    TABELA_LIMPA,
    USE_PHONETIC_STRIP_VOWELS,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
con = get_connection()
require_tables(con, [TABELA_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)

SPLINK_ANALYSIS_SAMPLE_N = 1_000_000
analysis_table = SPLINK_INPUT_VIEW
if n_reg > SPLINK_ANALYSIS_SAMPLE_N:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE splink_analysis_sample AS
    SELECT * FROM {SPLINK_INPUT_VIEW}
    USING SAMPLE {SPLINK_ANALYSIS_SAMPLE_N} ROWS
    ''')
    analysis_table = 'splink_analysis_sample'
    print(f'Amostra profile/blocking: {SPLINK_ANALYSIS_SAMPLE_N:,} de {n_reg:,}')


OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/censo_pessoas_2022_20260505.parquet
CENSO_CEP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/raw/censo/data_cep_uniq.csv
COHORT_DEDUP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/capefe/dados/CohortDados/cohort_dedup.parquet
FILTRO_UF: None
FILTRO_MUNICIPIO: 2111300
USE_PHONETIC_STRIP_VOWELS: False
ANO_OBITO_CORTE: 2021
ANO_NASCIMENTO_MIN: 1900
SEXO_VALIDOS: ('M', 'F')
DUCKDB_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/probabilistico.duckdb
splink_input → registro_limpo
Registros: 2,476,718
DuckDB: threads=20, memory_limit=279.3 GiB (defaults: 20, 300GB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Amostra profile/blocking: 1,000,000 de 2,476,718


## Sanity check - composição da base

Volume por fonte e preenchimento das colunas usadas no linkage. Blocking em
coluna muito vazia gera poucos pares candidatos.

In [2]:
from IPython.display import display

display(con.execute(f'''
SELECT origem, COUNT(*) AS n
FROM {SPLINK_INPUT_VIEW} GROUP BY 1 ORDER BY 2 DESC
''').df())

LINKAGE_COLS = [
    'primeiro_nome', 'ultimo_nome', 'nome_completo','nome_meio',
    'nome_mae', 'primeiro_nome_mae', 'nome_meio_mae', 'ultimo_nome_mae',
    'data_nascimento', 'idade', 'cep', 'sexo', 'uf',
    'nome_completo_phon','primeiro_nome_phon','nome_meio_phon','ultimo_nome_phon',
    'nome_mae_phon','primeiro_nome_mae_phon','nome_meio_mae_phon','ultimo_nome_mae_phon'
]

cols_presentes = [
    c for c in LINKAGE_COLS
    if c in set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)
]
preenchimento = ',\n    '.join(
    f"ROUND(100.0 * COUNT({c}) / COUNT(*), 1) AS pct_{c}" for c in cols_presentes
)
display(con.execute(f'''
SELECT origem, {preenchimento}
FROM {SPLINK_INPUT_VIEW} GROUP BY origem ORDER BY origem
''').df().T)


,origem,n
0,cpf,1438943
1,censo,1037775


,0,1
origem,censo,cpf
pct_primeiro_nome,95.6,100.0
pct_ultimo_nome,95.1,100.0
pct_nome_completo,95.6,100.0
pct_nome_meio,78.2,96.6
pct_nome_mae,28.9,95.8
pct_primeiro_nome_mae,28.9,95.8
pct_nome_meio_mae,23.6,89.6
pct_ultimo_nome_mae,28.9,95.8
pct_data_nascimento,83.3,97.7


## Exploração pré-modelo

Profile Splink das colunas de linkage e análise de blocking (cumulativo + maiores blocos).

In [3]:
from splink import block_on
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
from splink.exploratory import profile_columns
from splink.blocking_analysis import n_largest_blocks
# Blocking rules
blocking_rules = [
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'sexo'),
    block_on('ultimo_nome_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'data_nascimento'),
    block_on('cep', 'ultimo_nome_phon','sexo'),
    block_on('cep', 'primeiro_nome_phon','sexo'),    
]

profile_columns(
    con.execute(f'SELECT * FROM {analysis_table}').df(),
    db_api,
    column_expressions=[
        'primeiro_nome_phon', 'ultimo_nome_phon', 'nome_completo_phon',
        'cep', 'data_nascimento', 'idade',
    ],
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.VConcatChart(...)

In [4]:
# Análise de blocking só entre origens (link_only), na amostra ou na base completa.
con.execute(f"""
CREATE OR REPLACE VIEW splink_analysis_censo AS
SELECT * FROM {analysis_table} WHERE origem = 'censo'
""")
con.execute(f"""
CREATE OR REPLACE VIEW splink_analysis_cpf AS
SELECT * FROM {analysis_table} WHERE origem = 'cpf'
""")

cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=['splink_analysis_censo', 'splink_analysis_cpf'],
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type='link_only',
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.Chart(...)

## Modelo Splink

Settings e `Linker` em `link_only`: duas views (Censo / CPF) derivadas de
`SPLINK_INPUT_VIEW`, sem pares intra-base.


In [5]:
from splink import Linker, SettingsCreator
import splink.comparison_library as cl

input_cols = set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)

comparisons = [
    cl.NameComparison('nome_completo_phon',jaro_winkler_thresholds=[0.95]).configure(term_frequency_adjustments=True),
    cl.ExactMatch('primeiro_nome_phon').configure(term_frequency_adjustments=True),
    cl.ExactMatch('nome_meio_phon').configure(term_frequency_adjustments=True),
    cl.ExactMatch('ultimo_nome_phon').configure(term_frequency_adjustments=True),    
    cl.NameComparison('nome_mae_phon',jaro_winkler_thresholds=[0.95]).configure(term_frequency_adjustments=True),
    cl.ExactMatch('primeiro_nome_mae_phon').configure(term_frequency_adjustments=True),
    cl.ExactMatch('nome_meio_mae_phon').configure(term_frequency_adjustments=True),
    cl.ExactMatch('ultimo_nome_mae_phon').configure(term_frequency_adjustments=True),    
    cl.DateOfBirthComparison('data_nascimento', datetime_thresholds=[1,1,1], input_is_string=True,datetime_metrics=['month', 'year', 'year']),
    cl.ExactMatch('idade'),
    cl.ExactMatch('sexo').configure(term_frequency_adjustments=True),
    cl.ExactMatch('cep').configure(term_frequency_adjustments=True),
    #cl.ExactMatch('uf').configure(term_frequency_adjustments=True),    
]
if USE_PHONETIC_STRIP_VOWELS and 'nome_completo_phon_sv' in input_cols:
    comparisons.append(cl.NameComparison('nome_completo_phon_sv'))

con.execute(f"""
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
""")
con.execute(f"""
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
""")
print(
    'splink_censo:', con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0],
    '| splink_cpf:', con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0],
)

settings = SettingsCreator(
    link_type='link_only',
    unique_id_column_name='unique_id',
    comparisons=comparisons,
    blocking_rules_to_generate_predictions=blocking_rules,
    retain_intermediate_calculation_columns=True,
)
linker = Linker(
    ['splink_censo', 'splink_cpf'],
    settings,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)


In [6]:
deterministic_rules = [
    block_on('nome_completo_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'data_nascimento'),    
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)
linker.training.estimate_u_using_random_sampling(max_pairs=2_000_000)
linker.training.estimate_parameters_using_expectation_maximisation(block_on('data_nascimento'),estimate_without_term_frequencies=True)


Probability two random records match is estimated to be  2.37e-07.
This means that amongst all possible pairwise record comparisons, one in 4,217,668.99 are expected to match.  With 3,067,064,787,403 total possible comparisons, we expect a total of around 727,194.29 matching pairs
----- Estimating u probabilities using random sampling -----
u probability not trained for data_nascimento - Abs date difference <= 1 year (comparison vector value: 1). This usually means the comparison level was never observed in the training data.

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - nome_completo_phon (no m values are trained).
    - primeiro_nome_phon (no m values are trained).
    - nome_meio_phon (no m values are trained).
    - ultimo_nome_phon (no m values are trained).
    - nome_mae_phon (no m values are trained).
    - primeiro_nome_mae_phon (no m values are trained).
    - nome_meio_mae_phon (no m values are trained).
 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."data_nascimento" = r."data_nascimento"

Parameter estimates will be made for the following comparison(s):
    - nome_completo_phon
    - primeiro_nome_phon
    - nome_meio_phon
    - ultimo_nome_phon
    - nome_mae_phon
    - primeiro_nome_mae_phon
    - nome_meio_mae_phon
    - ultimo_nome_mae_phon
    - idade
    - sexo
    - cep

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - data_nascimento


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Iteration 1: Largest change in params was -0.655 in the m_probability of cep, level `Exact match on cep`
Iteration 2: Largest change in params was 0.172 in the m_probability of nome_completo_phon, level `All other comparisons`
Iteration 3: Largest change in params was 0.307 in the m_probability of nome_completo_phon, level `All other comparisons`
Iteration 4: Largest change in params was -0.39 in the m_probability of primeiro_nome_phon, level `Exact match on primeiro_nome_phon`
Iteration 5: Largest change in params was -0.348 in the m_probability of ultimo_nome_mae_phon, level `Exact match on ultimo_nome_mae_phon`
Iteration 6: Largest change in params was 0.438 in probability_two_random_records_match
Iteration 7: Largest change in params was 0.0372 in probability_two_random_records_match
Iteration 8: Largest change in params was 0.00275 in probability_two_random_records_match
Iteration 9: Largest change in params was 0.00258 in the m_probability of idade, level `All other comparisons`

<EMTrainingSession, blocking on l."data_nascimento" = r."data_nascimento", deactivating comparisons data_nascimento>

In [7]:
linker.training.estimate_parameters_using_expectation_maximisation(block_on('primeiro_nome', 'ultimo_nome','cep'),estimate_without_term_frequencies=True)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."primeiro_nome" = r."primeiro_nome") AND (l."ultimo_nome" = r."ultimo_nome") AND (l."cep" = r."cep")

Parameter estimates will be made for the following comparison(s):
    - nome_completo_phon
    - primeiro_nome_phon
    - nome_meio_phon
    - ultimo_nome_phon
    - nome_mae_phon
    - primeiro_nome_mae_phon
    - nome_meio_mae_phon
    - ultimo_nome_mae_phon
    - data_nascimento
    - idade
    - sexo

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - cep


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Level All other comparisons on comparison primeiro_nome_phon not observed in dataset, unable to train m value

Level All other comparisons on comparison ultimo_nome_phon not observed in dataset, unable to train m value

Level Abs date difference <= 1 year on comparison data_nascimento not observed in dataset, unable to train m value

Iteration 1: Largest change in params was 0.979 in the m_probability of primeiro_nome_phon, level `Exact match on primeiro_nome_phon`
Iteration 2: Largest change in params was -0.429 in the m_probability of data_nascimento, level `Exact match on date of birth`
Iteration 3: Largest change in params was 0.704 in the m_probability of data_nascimento, level `All other comparisons`
Iteration 4: Largest change in params was 0.487 in probability_two_random_records_match
Iteration 5: Largest change in params was 0.00101 in probability_two_random_records_match
Iteration 6: Largest change in params was 2.69e-07 in probability_two_random_records_match

EM converged 

<EMTrainingSession, blocking on (l."primeiro_nome" = r."primeiro_nome") AND (l."ultimo_nome" = r."ultimo_nome") AND (l."cep" = r."cep"), deactivating comparisons cep>

In [8]:
SPLINK_MODEL_JSON.parent.mkdir(parents=True, exist_ok=True)
linker.misc.save_model_to_json(str(SPLINK_MODEL_JSON), overwrite=True)
print('Modelo salvo:', SPLINK_MODEL_JSON)

Modelo salvo: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_model.json


## Visualização pós-treino

Match weights e registros difíceis de linkar (unlinkables).

In [9]:
linker.visualisations.match_weights_chart()


/opt/venvs/singed/jhub/lib64/python3.9/site-packages/altair/vegalite/v6/api.py:4124: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [10]:
linker.evaluation.unlinkables_chart()

alt.LayerChart(...)

## Predict + clustering

In [11]:
df_predict = linker.inference.predict(threshold_match_probability=0.5)
df_predictions = df_predict.as_pandas_dataframe()
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=0.95,
)
df_clusters = clusters.as_pandas_dataframe()
print('Pares:', len(df_predictions), 'Clusters:', df_clusters['cluster_id'].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Blocking time: 104.93 seconds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Predict time: 401.10 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'data_nascimento':
    m values not fully trained
Comparison: 'data_nascimento':
    u values not fully trained


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Completed iteration 2, num edges remaining to process: 57524
Completed iteration 3, num edges remaining to process: 10086
Completed iteration 4, num edges remaining to process: 2032
Completed iteration 5, num edges remaining to process: 428
Completed iteration 6, num edges remaining to process: 66
Completed iteration 7, num edges remaining to process: 8
Completed iteration 8, num edges remaining to process: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pares: 2337583 Clusters: 1538243


## Waterfall — amostra de pares

Contribuição de cada comparação ao match weight (5 pares).

In [12]:
records_to_plot = df_predictions.head(5).to_dict(orient="records")
linker.visualisations.waterfall_chart(records_to_plot, filter_nulls=False)


#records_sample = df_predictions.to_dict(orient='records')
#Linker.visualisations.waterfall_chart(records_sample, filter_nulls=False)

alt.LayerChart(...)

In [23]:
pd.set_option('display.max_columns', None)
#df_predictions.head(5)
#df_predictions['unique_id_l'].str.contains("censo").head(5)
df_predictions[(df_predictions['unique_id_l'].str.contains("censo")) & df_predictions['unique_id_r'].str.contains("cpf")].head(5)




,match_weight,match_probability,unique_id_l,unique_id_r,nome_completo_phon_l,nome_completo_phon_r,gamma_nome_completo_phon,tf_nome_completo_phon_l,tf_nome_completo_phon_r,bf_nome_completo_phon,bf_tf_adj_nome_completo_phon,primeiro_nome_phon_l,primeiro_nome_phon_r,gamma_primeiro_nome_phon,tf_primeiro_nome_phon_l,tf_primeiro_nome_phon_r,bf_primeiro_nome_phon,bf_tf_adj_primeiro_nome_phon,nome_meio_phon_l,nome_meio_phon_r,gamma_nome_meio_phon,tf_nome_meio_phon_l,tf_nome_meio_phon_r,bf_nome_meio_phon,bf_tf_adj_nome_meio_phon,ultimo_nome_phon_l,ultimo_nome_phon_r,gamma_ultimo_nome_phon,tf_ultimo_nome_phon_l,tf_ultimo_nome_phon_r,bf_ultimo_nome_phon,bf_tf_adj_ultimo_nome_phon,nome_mae_phon_l,nome_mae_phon_r,gamma_nome_mae_phon,tf_nome_mae_phon_l,tf_nome_mae_phon_r,bf_nome_mae_phon,bf_tf_adj_nome_mae_phon,primeiro_nome_mae_phon_l,primeiro_nome_mae_phon_r,gamma_primeiro_nome_mae_phon,tf_primeiro_nome_mae_phon_l,tf_primeiro_nome_mae_phon_r,bf_primeiro_nome_mae_phon,bf_tf_adj_primeiro_nome_mae_phon,nome_meio_mae_phon_l,nome_meio_mae_phon_r,gamma_nome_meio_mae_phon,tf_nome_meio_mae_phon_l,tf_nome_meio_mae_phon_r,bf_nome_meio_mae_phon,bf_tf_adj_nome_meio_mae_phon,ultimo_nome_mae_phon_l,ultimo_nome_mae_phon_r,gamma_ultimo_nome_mae_phon,tf_ultimo_nome_mae_phon_l,tf_ultimo_nome_mae_phon_r,bf_ultimo_nome_mae_phon,bf_tf_adj_ultimo_nome_mae_phon,data_nascimento_l,data_nascimento_r,gamma_data_nascimento,bf_data_nascimento,idade_l,idade_r,gamma_idade,bf_idade,sexo_l,sexo_r,gamma_sexo,tf_sexo_l,tf_sexo_r,bf_sexo,bf_tf_adj_sexo,cep_l,cep_r,gamma_cep,tf_cep_l,tf_cep_r,bf_cep,bf_tf_adj_cep,match_key
0,1.688735,0.763243,censo_2111300050009520070040000436000001000000002,cpf_78057876387,RONALDO PEREIRA SANTOS,RONALDO PEREIRA SANTOS,2,2.879462e-06,2.879462e-06,8069.904625,0.498013,RONALDO,RONALDO,1,0.000807,0.000807,55.268153,11.434435,PEREIRA,PEREIRA,1,1.611989e-02,0.016120,3.203332,0.288506,SANTOS,SANTOS,1,0.070248,0.070248,21.005005,0.349767,None,MARIA ROSARIO PEREIRA SANTOS,-1,NaN,9.533400e-06,1.000000,1.0,None,MARIA,-1,NaN,0.230407,1.000000,1.0,None,ROSARIO PEREIRA,-1,NaN,0.000141,1.000000,1.000000,None,SANTOS,-1,NaN,0.076886,1.000000,1.000000,None,1978-10-08,-1,1.000000,54,44,0,0.513988,M,M,1,0.480315,0.480315,1.501661,1.040886,65045292,65092408,0,0.000198,0.000020,0.981401,1.0,0
3,0.697643,0.618590,censo_2111300050010010040020000065000001000000002,cpf_00362079340,PAULO ANDERSON KAMARA RIBEIRO,PAULO VIKTOR KAMARA RIBEIRO,0,8.227033e-07,8.227033e-07,0.985564,1.000000,PAULO,PAULO,1,0.006470,0.006470,55.268153,1.427123,ANDERSON KAMARA,VIKTOR KAMARA,0,9.087002e-07,0.000007,0.989705,1.000000,RIBEIRO,RIBEIRO,1,0.013776,0.013776,21.005005,1.783578,ALEXANDRINA KRISTINA KAMARA RIBEIRO,MARIA KRISTINA KAMARA RIBEIRO,0,0.000002,1.191675e-06,0.995343,1.0,ALEXANDRINA,MARIA,0,0.000509,0.230407,0.973712,1.0,KRISTINA KAMARA,KRISTINA KAMARA,1,0.000035,0.000035,2.191262,142.355951,RIBEIRO,RIBEIRO,1,0.013876,0.013876,4.705276,2.177031,1995-11-22,1983-09-23,0,0.971951,26,39,0,0.513988,M,M,1,0.480315,0.480315,1.501661,1.040886,65058145,65075020,0,0.000131,0.000113,0.981401,1.0,0
5,0.235375,0.540697,censo_2111300050009520010120000340000001000000002,cpf_78869978320,NATANAEL SANTOS SILVA,NATANAEL SANTOS SILVA,2,4.524868e-06,4.524868e-06,8069.904625,0.316918,NATANAEL,NATANAEL,1,0.000538,0.000538,55.268153,17.147284,SANTOS,SANTOS,1,2.839506e-02,0.028395,3.203332,0.163785,SILVA,SILVA,1,0.101296,0.101296,21.005005,0.242563,None,ROSA IRENE SANTOS SILVA,-1,NaN,2.979187e-06,1.000000,1.0,None,ROSA,-1,NaN,0.006535,1.000000,1.0,None,IRENE SANTOS,-1,NaN,0.000020,1.000000,1.000000,None,SILVA,-1,NaN,0.112581,1.000000,1.000000,2000-03-18,1976-12-27,0,0.971951,22,46,0,0.513988,M,M,1,0.480315,0.480315,1.501661,1.040886,65045292,65030005,0,0.000198,0.000156,0.981401,1.0,0
6,4.577593,0.959803,censo_2111300050009510010010000005000001000504001,cpf_09562567320,MANOEL KRUS FEREIRA,MANOEL KRUS FEREIRA,2,2.056758e-06,2.056758e-06,8069.904625,0.697219,MANOEL,MANOEL,1,0.003509,0.003509,55.268153,2.6

## Cluster studio

Dashboard interativo para inspecionar clusters (amostra por tamanho).

In [14]:
from IPython.display import IFrame, display

CLUSTER_STUDIO_HTML = OUTPUT_DIR / 'dashboards' / 'cluster_studio.html'
CLUSTER_STUDIO_HTML.parent.mkdir(parents=True, exist_ok=True)

linker.visualisations.cluster_studio_dashboard(
    df_predict,
    clusters,
    str(CLUSTER_STUDIO_HTML),
    sampling_method='by_cluster_size',
    overwrite=True,
)
print('Dashboard:', CLUSTER_STUDIO_HTML)
display(IFrame(src=str(CLUSTER_STUDIO_HTML), width='100%', height=1200))

Dashboard: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/dashboards/cluster_studio.html


## Diagnóstico dos scores (sem rótulos)

Distribuição de match weight e tamanho de cluster. Precision/recall contra a
coorte ficam no NB03, sobre o modelo salvo acima.

In [15]:
display(df_predictions['match_probability'].describe())
faixas = pd.cut(df_predictions['match_weight'], bins=20)
display(
    df_predictions.groupby(faixas, observed=True)
    .size()
    .rename('n_pares')
    .to_frame()
)

count    2.337583e+06
mean     8.665328e-01
std      1.624563e-01
min      5.000020e-01
25%      7.435105e-01
50%      9.581401e-01
75%      9.999901e-01
max      1.000000e+00
Name: match_probability, dtype: float64

,n_pares
match_weight,
"(-0.108, 5.407]",1271020
"(5.407, 10.815]",320878
"(10.815, 16.222]",151014
"(16.222, 21.629]",134280
"(21.629, 27.036]",127775
"(27.036, 32.444]",105610
"(32.444, 37.851]",74308
"(37.851, 43.258]",48854
"(43.258, 48.666]",33419


In [16]:
tamanhos = df_clusters.groupby('cluster_id').size()
print(f'Clusters: {tamanhos.size:,} | maior: {tamanhos.max():,} | singletons: {(tamanhos == 1).sum():,}')
display(
    tamanhos.value_counts().sort_index().head(20)
    .rename('n_clusters').to_frame().rename_axis('tamanho')
)

# Clusters grandes demais indicam blocking/threshold frouxo — inspecionar antes do NB03.
display(tamanhos.sort_values(ascending=False).head(10).rename('tamanho').to_frame())

Clusters: 1,538,243 | maior: 790 | singletons: 936,955


,n_clusters
tamanho,
1,936955
2,458653
3,63148
4,44162
5,12957
6,9462
7,4019
8,2713
9,1661


,tamanho
cluster_id,
censo_2111300050000100080040000234000001000000001,790
censo_2111300050000220010020000004000002000000004,340
censo_2111300050000690020020000073000001000000001,250
censo_2111300050000090090010000480000001000000005,245
censo_2111300050000410030030000325000001000000003,211
censo_2111300050000060130060000455000001000000001,165
censo_2111300050000460030030000152000001000000002,157
censo_2111300050000120180020000431000001000000002,148
censo_2111300050018010010200000120000001000000003,137


In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_predictions.to_parquet(SPLINK_PREDICTIONS)
df_clusters.to_parquet(SPLINK_CLUSTERS)
print('Predictions:', SPLINK_PREDICTIONS)
print('Clusters:', SPLINK_CLUSTERS)

Predictions: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_predictions.parquet
Clusters: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_clusters.parquet


## Encerrar

Artefatos prontos para o [`03_validar_coorte.ipynb`](03_validar_coorte.ipynb).

In [18]:
for label, p in [
    ('modelo', SPLINK_MODEL_JSON),
    ('predictions', SPLINK_PREDICTIONS),
    ('clusters', SPLINK_CLUSTERS),
]:
    print(f'{label:12s} {"ok " if p.exists() else "FALTA"} {p}')

con.close()


modelo       ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_model.json
predictions  ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_predictions.parquet
clusters     ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_clusters.parquet


In [19]:
con.close()